In [ ]:
from sklearn.decomposition import PCA
from matplotlib import pyplot as plt
import pandas as pd
from sklearn import metrics
import time

def pca(df_X, df_y):
    pca = PCA(n_components = 2)
    pca.fit(df_X)
    df_pca = pca.transform(df_X)
    df_pca = pd.DataFrame(df_pca, columns = ['comp. 0', 'comp. 1'])
    df_pca['target'] = df_y
    print('variance ratio:', pca.explained_variance_ratio_, 'sum:', sum(pca.explained_variance_ratio_))
    return df_pca

def concat(df_X, df_y):
    df = pd.concat([df_X, df_y], axis=1)
    return df

def concath(df_X, df_y):
    df = pd.concat([df_X, df_y])
    return df

###################################### ADALINE #############################################    
def linear(X,theta):
    z = np.dot(X,theta.T)
    return z

def cost_ada(z,y):              # SSE
    loss = ((y-z)**2).sum()
    return loss/2.0

def gd_ada(X,z,y):              # SSE
    return -np.dot((y-z), X)

def cost_ada2(z,y):              # MSE
    loss = ((y-z)**2).sum()
    return loss/len(y)

def gd_ada2(X,z,y):               # MSE
    return -np.dot((y-z), X)/len(y)

def update_loss(theta,learning_rate,gradient):
    return theta-(learning_rate*gradient)    
    
def predict_ada(X,theta):
    outcome = []
    result = linear(X,theta)
    for i in range(X.shape[0]):
        if result[i] <= 0:
            outcome.append(-1)
        else:
            outcome.append(1)
    return outcome    
    
def plot_cost_function(cost):
    plt.plot(cost,label="loss")
    plt.ylim(-0.1,1.1)
    plt.xlabel('Iteration',fontweight="bold",fontsize = 15)
    plt.ylabel('Loss',fontweight="bold",fontsize = 15)
    plt.title("Cost Function",fontweight="bold",fontsize = 20)
    plt.legend()
    plt.show()    

###################################### BCE & Ours ############################################# 

def predict(X,theta):
    outcome = []
    result = sigmoid(X,theta)
    for i in range(X.shape[0]):
        if result[i] <= threshold:
            outcome.append(0)
        else:
            outcome.append(1)
    return outcome  

def predict_ours(X,theta):
    outcome = []
    result = sigmoid_2(sigmoid(X,theta))
    for i in range(X.shape[0]):
        if result[i] <= threshold:
            outcome.append(0)
        else:
            outcome.append(1)
    return outcome    

def sigmoid(X,theta):
    z = np.dot(X,theta.T).astype(float)
    return 1.0/(1+np.exp(-z))

def sigmoid_2(p):
    s = (L*(p-0.5)).astype(float)
    return 1.0/(1+np.exp(-s))

def cost_function(h,y):
    loss = ((-y * np.log(h))-((1-y)* np.log(1-h))).mean()
    return loss

def gradient_descent(X,h,y,yl):
    return np.dot(X.T,(h-y))/yl

def cost_function_new(bs,syh,syhy,bs_sy):
    loss = (1+bs)*syhy / ( bs_sy + syh )  # f_score
    return 1-loss

def gradient_descent_new(p,X,yh,y,bs,syh,syhy,bs_sy):
    yp_pz = L*yh*(1-yh) * p*(1-p)
    return -( (1+bs) * (np.dot(y*yp_pz*(bs_sy+syh), X) - np.dot(yp_pz*syhy, X)) ) / ( (bs_sy + syh)**2 )

def cost_function_acc(sy,syh,syhy,yl):
    loss = (yl-sy-syh+2*syhy)/yl   # accyracy
    return 1-loss

def gradient_descent_acc(p,X,yh,y,yl):
    yp_pz = L*yh*(1-yh) * p*(1-p)
    return (np.dot(yp_pz, X) - 2*np.dot(y*yp_pz, X)) / yl

def cost_function_pre(syh,syhy):
    loss = syhy/syh  # precision
    return 1-loss

def gradient_descent_pre(p,X,yh,y,syh,syhy):
    yp_pz = L*yh*(1-yh) * p*(1-p)
    return (-np.dot(y*yp_pz, X)*syh + np.dot(yp_pz, X)*syhy) / (syh**2)

def cost_function_rec(sy,syhy):
    loss = syhy/sy  # recall
    return 1-loss

def gradient_descent_rec(p,X,yh,y,sy):
    yp_pz = L*yh*(1-yh) * p*(1-p)
    return -np.dot(y*yp_pz, X) / sy

def cost_function_gmean(sy,syh,syhy,yl):
    loss = (syhy*(yl-syh-sy+syhy)/(sy*(yl-sy)))**0.5  # gmean
    return 1-loss

def gradient_descent_gmean(p,X,yh,y,sy,syh,syhy,yl):
    yp_pz = L*yh*(1-yh) * p*(1-p)
    repeat1 = np.dot(y*yp_pz, X)
    repeat2 = yl-syh-sy+syhy
    return -2*( (repeat1*repeat2) + (-np.dot(yp_pz, X)+repeat1)*syhy ) / (sy*(yl-sy)*syhy*(repeat2))**0.5

def cost_function_balacc(sy,syh,syhy,yl):
    loss = (yl*(syhy+sy)-sy*(syh+sy)) / (2*sy*(yl-sy))     # balanced accuracy
    return 1-loss

def gradient_descent_balacc(p,X,yh,y,sy,yl):
    yp_pz = L*yh*(1-yh) * p*(1-p)
    return -(yl*np.dot(y*yp_pz, X)-sy*np.dot(yp_pz, X))/(2*sy*(yl-sy))

# 1. My own data(2d / 10,000)

In [ ]:
from sklearn import datasets
import numpy as np
import pandas as pd
Init_X, Init_y = datasets.make_classification(n_samples=10000, n_classes=2, weights=[0.9, 0.1], class_sep=1.2,
                                    n_features=5, n_informative=3, n_redundant=1, n_clusters_per_class=1, random_state=0)
X = np.array(Init_X)
y = np.array(Init_y)
# change 0 -> -1
y = [-1 if x==0 else x for x in y]

df_pca = pca(X, y)
df_pca

In [ ]:
TT_mse= []
TT_bce= []
TT_f1= []
TT_gmean= []
TT_bacc= []

In [ ]:
# train = train data // test = validation data // 10 cross-validation(train 0.9, validate 0.1, 10 times)
# The average of test results are the validation scores.

from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state = 2)
n_iter=0

X = df_pca.iloc[:, :2]
y = df_pca.iloc[:, 2]

for train_index, test_index in skf.split(df_pca, df_pca['target']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
#     print('Labels for train:\n', label_train.value_counts())
#     print('Labels for test:\n', label_test.value_counts())
#     print(len(X_train), len(y_train), len(X_test), len(y_test))
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    X_test = np.array(X_test)
    y_test = np.array(y_test)    
    intercept_r = np.ones((X_train.shape[0],1))
    intercept_t = np.ones((X_test.shape[0],1))
    X_train_a = np.concatenate((intercept_r,X_train),axis= 1)
    X_test_a = np.concatenate((intercept_t,X_test),axis= 1)
    
###################### MSE (with Sigmoid)##############################
    start_mse = time.time()
    theta = np.zeros(X_train_a.shape[1])
    threshold = 0.5
    num_iter = 1000
    cost_mse = []
    lr = 0.1
    yl = y_train.shape[0]
    for i in range(num_iter):
        h = sigmoid(X_train_a,theta)
        cost_mse.append(cost_ada2(h,y_train))
        gradient = gd_ada2(X_train_a,h,y_train)
        theta = update_loss(theta,lr,gradient)
    TT_mse.append(cost_mse)
        
###################### Label change (-1 -> 0) ############################## 
    y_train = [0 if x==-1 else x for x in y_train]
    y_train = np.array(y_train)
    y_test = [0 if x==-1 else x for x in y_test]
    y_test = np.array(y_test)
    
###################### BCE (with Sigmoid) ############################## 
    start_bce = time.time()
    theta = np.zeros(X_train_a.shape[1])
    threshold = 0.5
    num_iter = 1000
    cost_bce = []
    lr = 0.2
    yl = y_train.shape[0]
    for i in range(num_iter):
        h = sigmoid(X_train_a,theta)
        cost_bce.append(cost_function(h,y_train))
        gradient = gradient_descent(X_train_a,h,y_train,yl)
        theta = update_loss(theta,lr,gradient)
    TT_bce.append(cost_bce)
    
###################### Ours(F1) ############################## 
    start_f1 = time.time()
    theta = np.zeros(X_train_a.shape[1])
    num_iter = 1000
    lr = 0.01
    beta = 1            
    L = 73               
    cost_f1 = []
    bs = beta**2
    sy = np.sum(y_train)
    bs_sy = bs*sy
    for i in range(num_iter):
        p = sigmoid(X_train_a,theta)
        yh = sigmoid_2(p)                                          
        syh = np.sum(yh)
        syhy = np.dot(yh,y_train)
        cost_f1.append(cost_function_new(bs,syh,syhy,bs_sy))                
        gradient = gradient_descent_new(p,X_train_a,yh,y_train,bs,syh,syhy,bs_sy)      
        theta = update_loss(theta,lr,gradient)
    TT_f1.append(cost_f1)
    
###################### Ours(Gmean) ##############################
    start_gmean = time.time()
    theta = np.zeros(X_train_a.shape[1])
    num_iter = 1000
    lr = 0.01          
    L = 73               
    cost_gmean = []
    sy = np.sum(y_train)
    yl = y_train.shape[0]
    for i in range(num_iter):
        p = sigmoid(X_train_a,theta)
        yh = sigmoid_2(p)                                          
        syh = np.sum(yh)
        syhy = np.dot(yh,y_train)
        cost_gmean.append(cost_function_gmean(sy,syh,syhy,yl))                
        gradient = gradient_descent_gmean(p,X_train_a,yh,y_train,sy,syh,syhy,yl)      
        theta = update_loss(theta,lr,gradient)
    TT_gmean.append(cost_gmean)
    
###################### Ours(Balanced Accuracy) ##############################
    start_bacc = time.time()
    theta = np.zeros(X_train_a.shape[1])
    num_iter = 1000
    lr = 0.01
    L = 73               
    cost_bacc = []
    sy = np.sum(y_train)
    yl = y_train.shape[0]
    for i in range(num_iter):
        p = sigmoid(X_train_a,theta)
        yh = sigmoid_2(p)                                          
        syh = np.sum(yh)
        syhy = np.dot(yh,y_train)
        cost_bacc.append(cost_function_balacc(sy,syh,syhy,yl))                
        gradient = gradient_descent_balacc(p,X_train_a,yh,y_train,sy,yl)     
        theta = update_loss(theta,lr,gradient)
    TT_bacc.append(cost_bacc)

In [ ]:
tts = [TT_mse, TT_bce, TT_f1, TT_gmean, TT_bacc]
cost_mse=[]
cost_bce=[]
cost_f1=[]
cost_gmean=[]
cost_bacc=[]
ars = [cost_mse, cost_bce, cost_f1, cost_gmean, cost_bacc]
for k in range(5):
    for i in range(1000):
        suum = 0
        for j in range(10):
            suum += tts[k][j][i]
        ars[k].append(suum/10)

In [ ]:
print("L\u00b2")
print("L\u2081")

In [ ]:
# plt.rcParams["figure.figsize"] = [5, 3]
plt.rcParams["figure.dpi"] = 200
plt.rcParams['axes.titlesize'] = 20  
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['axes.labelsize'] = 20  
plt.rcParams['font.size'] = 15

# plt.plot(cost_mse,label="MSE", color='green')
plt.plot(cost_bce,label="BCE", color='red', linewidth = 3)
plt.plot(cost_f1,label="$L_{F1}$", color='deepskyblue', ls='-', linewidth = 3)
plt.plot(cost_gmean,label="$L_{G}$", color='deepskyblue', ls=':', linewidth = 3)
plt.plot(cost_bacc,label="$L_{B}$", color='deepskyblue', ls='--', linewidth = 3)

plt.ylim(-0.1,1.1)
plt.xlabel('Iteration',fontsize = 20)
plt.ylabel('Loss',fontsize = 20)
plt.title("Loss Curve (Data#1)",fontsize = 20)
plt.legend()
plt.show()  

# 2. Creditcard Fraud Detection 2023(29d / 298531)

In [ ]:
# class '0' = normal, class '1' = anomaly
card_df = pd.read_csv('creditcard_2023.csv')
card_df.shape

In [ ]:
card_df.isnull().sum()

In [ ]:
card_df.head()

In [ ]:
card_df.describe()

In [ ]:
# Amount values largely varies.

# # Normalization
# card_df.iloc[:,:-1] = (card_df.iloc[:,:-1] - card_df.iloc[:,:-1].min())/(card_df.iloc[:,:-1].max() - card_df.iloc[:,:-1].min())

# Standardization
card_df.iloc[:,:-1] = (card_df.iloc[:,:-1] - card_df.iloc[:,:-1].mean())/card_df.iloc[:,:-1].std()

card_df

In [ ]:
card_df['Class'].value_counts()

In [ ]:
# Data is too balanced!!! We intentionally make it imbalanced.
df_0 = card_df[card_df['Class']==0]
df_1 = card_df[card_df['Class']==1]
print(len(df_0), len(df_1))

In [ ]:
N = round(len(df_0)*0.05)
df_1_samp = df_1.sample(n=N, random_state = 100)
df_1_samp

In [ ]:
df_card = concath(df_0, df_1_samp)
df_card

In [ ]:
df_card.columns

In [ ]:
df_card = df_card.drop('id', axis=1)
df_card

In [ ]:
df_card['Class'].value_counts()

In [ ]:
TT_mse= []
TT_bce= []
TT_f1= []
TT_gmean= []
TT_bacc= []

In [ ]:
# train = train data // test = validation data // 10 cross-validation(train 0.9, validate 0.1, 10 times)
# The average of test results are the validation scores.

from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state = 2)
n_iter=0

X = df_card.iloc[:, :-1]
y = df_card.iloc[:, -1]

for train_index, test_index in skf.split(df_card, df_card['Class']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
#     print('Labels for train:\n', label_train.value_counts())
#     print('Labels for test:\n', label_test.value_counts())
#     print(len(X_train), len(y_train), len(X_test), len(y_test))
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    X_test = np.array(X_test)
    y_test = np.array(y_test)    
    intercept_r = np.ones((X_train.shape[0],1))
    intercept_t = np.ones((X_test.shape[0],1))
    X_train_a = np.concatenate((intercept_r,X_train),axis= 1)
    X_test_a = np.concatenate((intercept_t,X_test),axis= 1)
        
###################### Label change (-1 -> 0) ############################## 
    y_train = [0 if x==-1 else x for x in y_train]
    y_train = np.array(y_train)
    y_test = [0 if x==-1 else x for x in y_test]
    y_test = np.array(y_test)
    
###################### BCE (with Sigmoid) ############################## 
    start_bce = time.time()
    theta = np.zeros(X_train_a.shape[1])
    threshold = 0.5
    num_iter = 1000
    cost_bce = []
    lr = 0.01
    yl = y_train.shape[0]
    for i in range(num_iter):
        h = sigmoid(X_train_a,theta)
        cost_bce.append(cost_function(h,y_train))
        gradient = gradient_descent(X_train_a,h,y_train,yl)
        theta = update_loss(theta,lr,gradient)
#     plot_cost_function(cost)
#     print(cost[0]/cost[0], cost[99]/cost[0], cost[199]/cost[0], cost[299]/cost[0], cost[399]/cost[0]
#          , cost[499]/cost[0], cost[599]/cost[0], cost[699]/cost[0], cost[799]/cost[0], 
#           cost[899]/cost[0], cost[999]/cost[0])
    TT_bce.append(cost_bce)
    
###################### Ours(F1) ############################## 
    start_f1 = time.time()
    theta = np.zeros(X_train_a.shape[1])
    num_iter = 1000
    lr = 0.005
    beta = 1            
    L = 73               
    cost_f1 = []
    bs = beta**2
    sy = np.sum(y_train)
    bs_sy = bs*sy
    for i in range(num_iter):
        p = sigmoid(X_train_a,theta)
        yh = sigmoid_2(p)                                          
        syh = np.sum(yh)
        syhy = np.dot(yh,y_train)
        cost_f1.append(cost_function_new(bs,syh,syhy,bs_sy))                
        gradient = gradient_descent_new(p,X_train_a,yh,y_train,bs,syh,syhy,bs_sy)      
        theta = update_loss(theta,lr,gradient)
#     plot_cost_function(cost)
#     print(cost[0]/cost[0], cost[99]/cost[0], cost[199]/cost[0], cost[299]/cost[0], cost[399]/cost[0]
#          , cost[499]/cost[0], cost[599]/cost[0], cost[699]/cost[0], cost[799]/cost[0], 
#           cost[899]/cost[0], cost[999]/cost[0])
    TT_f1.append(cost_f1)
    
###################### Ours(Gmean) ##############################
    start_gmean = time.time()
    theta = np.zeros(X_train_a.shape[1])
    num_iter = 1000
    lr = 0.001          
    L = 73               
    cost_gmean = []
    sy = np.sum(y_train)
    yl = y_train.shape[0]
    for i in range(num_iter):
        p = sigmoid(X_train_a,theta)
        yh = sigmoid_2(p)                                          
        syh = np.sum(yh)
        syhy = np.dot(yh,y_train)
        cost_gmean.append(cost_function_gmean(sy,syh,syhy,yl))                
        gradient = gradient_descent_gmean(p,X_train_a,yh,y_train,sy,syh,syhy,yl)      
        theta = update_loss(theta,lr,gradient)
#     plot_cost_function(cost)
#     print(cost[0]/cost[0], cost[99]/cost[0], cost[199]/cost[0], cost[299]/cost[0], cost[399]/cost[0]
#          , cost[499]/cost[0], cost[599]/cost[0], cost[699]/cost[0], cost[799]/cost[0], 
#           cost[899]/cost[0], cost[999]/cost[0])
    TT_gmean.append(cost_gmean)
    
###################### Ours(Balanced Accuracy) ##############################
    start_bacc = time.time()
    theta = np.zeros(X_train_a.shape[1])
    num_iter = 1000
    lr = 0.005
    L = 73               
    cost_bacc = []
    sy = np.sum(y_train)
    yl = y_train.shape[0]
    for i in range(num_iter):
        p = sigmoid(X_train_a,theta)
        yh = sigmoid_2(p)                                          
        syh = np.sum(yh)
        syhy = np.dot(yh,y_train)
        cost_bacc.append(cost_function_balacc(sy,syh,syhy,yl))                
        gradient = gradient_descent_balacc(p,X_train_a,yh,y_train,sy,yl)     
        theta = update_loss(theta,lr,gradient)
#     plot_cost_function(cost)
#     print(cost[0]/cost[0], cost[99]/cost[0], cost[199]/cost[0], cost[299]/cost[0], cost[399]/cost[0]
#          , cost[499]/cost[0], cost[599]/cost[0], cost[699]/cost[0], cost[799]/cost[0], 
#           cost[899]/cost[0], cost[999]/cost[0])
    TT_bacc.append(cost_bacc)

In [ ]:
tts = [TT_bce, TT_f1, TT_gmean, TT_bacc]
cost_bce=[]
cost_f1=[]
cost_gmean=[]
cost_bacc=[]
ars = [cost_bce, cost_f1, cost_gmean, cost_bacc]
for k in range(4):
    for i in range(1000):
        suum = 0
        for j in range(10):
            suum += tts[k][j][i]
        ars[k].append(suum/10)

In [ ]:
# plt.rcParams["figure.figsize"] = [5, 3]
plt.rcParams["figure.dpi"] = 200
plt.rcParams['axes.titlesize'] = 20  
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['axes.labelsize'] = 20  
plt.rcParams['font.size'] = 15

# plt.plot(cost_mse,label="MSE", color='green')
plt.plot(cost_bce,label="BCE", color='red', linewidth = 3)
plt.plot(cost_f1,label="$L_{F1}$", color='deepskyblue', ls='-', linewidth = 3)
plt.plot(cost_gmean,label="$L_{G}$", color='deepskyblue', ls=':', linewidth = 3)
plt.plot(cost_bacc,label="$L_{B}$", color='deepskyblue', ls='--', linewidth = 3)

plt.ylim(-0.1,1.1)
plt.xlabel('Iteration',fontsize = 20)
plt.ylabel('Loss',fontsize = 20)
plt.title("Loss Curve (Data#2)",fontsize = 20)
plt.legend()
plt.show()  

# 3. Breast Cancer Data (30d / 569)

In [ ]:
# class 'B' = Benign, class 'M' = Malignant
cancer_df = pd.read_csv('breast_cancer.csv')
cancer_df.shape

In [ ]:
cancer_df.isnull().sum()

In [ ]:
cancer_df.head()

In [ ]:
cancer_df.describe()

In [ ]:
# M/Malignant = 0, B/Benign = 1
y_encoded, y_class = pd.factorize(cancer_df['diagnosis'])
print(y_class)
y_encoded

In [ ]:
# But I want [B/Benign = 0(Major), M/Malignant = 1(minor)]
y_encoded = (y_encoded+1)%2
y_encoded

In [ ]:
cancer_df['label'] = y_encoded
cancer_df

In [ ]:
cancer_df = cancer_df.drop('id', axis=1)
cancer_df = cancer_df.drop('diagnosis', axis=1)
cancer_df

In [ ]:
# Amount values largely varies.

# # Normalization
# card_df.iloc[:,:-1] = (card_df.iloc[:,:-1] - card_df.iloc[:,:-1].min())/(card_df.iloc[:,:-1].max() - card_df.iloc[:,:-1].min())

# Standardization
cancer_df.iloc[:,:-1] = (cancer_df.iloc[:,:-1] - cancer_df.iloc[:,:-1].mean())/cancer_df.iloc[:,:-1].std()

cancer_df

In [ ]:
cancer_df['label'].value_counts()

In [ ]:
# Data is too balanced!!! We intentionally make it imbalanced.
df_0 = cancer_df[cancer_df['label']==0]
df_1 = cancer_df[cancer_df['label']==1]
print(len(df_0), len(df_1))

In [ ]:
N = round(len(df_0)*0.1)
df_1_samp = df_1.sample(n=N, random_state = 100)
df_1_samp

In [ ]:
cancer_df = concath(df_0, df_1_samp)
cancer_df

In [ ]:
cancer_df['label'].value_counts()

In [ ]:
TT_mse= []
TT_bce= []
TT_f1= []
TT_gmean= []
TT_bacc= []

In [ ]:
# train = train data // test = validation data // 10 cross-validation(train 0.9, validate 0.1, 10 times)
# The average of test results are the validation scores.

from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state = 2)
n_iter=0

X = cancer_df.iloc[:, :-1]
y = cancer_df.iloc[:, -1]


for train_index, test_index in skf.split(cancer_df, cancer_df['label']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
#     print('Labels for train:\n', label_train.value_counts())
#     print('Labels for test:\n', label_test.value_counts())
#     print(len(X_train), len(y_train), len(X_test), len(y_test))
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    X_test = np.array(X_test)
    y_test = np.array(y_test)    
    intercept_r = np.ones((X_train.shape[0],1))
    intercept_t = np.ones((X_test.shape[0],1))
    X_train_a = np.concatenate((intercept_r,X_train),axis= 1)
    X_test_a = np.concatenate((intercept_t,X_test),axis= 1)
        
###################### Label change (-1 -> 0) ############################## 
    y_train = [0 if x==-1 else x for x in y_train]
    y_train = np.array(y_train)
    y_test = [0 if x==-1 else x for x in y_test]
    y_test = np.array(y_test)
    
###################### BCE (with Sigmoid) ############################## 
    start_bce = time.time()
    theta = np.zeros(X_train_a.shape[1])
    threshold = 0.5
    num_iter = 1000
    cost_bce = []
    lr = 0.05
    yl = y_train.shape[0]
    for i in range(num_iter):
        h = sigmoid(X_train_a,theta)
        cost_bce.append(cost_function(h,y_train))
        gradient = gradient_descent(X_train_a,h,y_train,yl)
        theta = update_loss(theta,lr,gradient)
#     plot_cost_function(cost)
#     print(cost[0]/cost[0], cost[99]/cost[0], cost[199]/cost[0], cost[299]/cost[0], cost[399]/cost[0]
#          , cost[499]/cost[0], cost[599]/cost[0], cost[699]/cost[0], cost[799]/cost[0], 
#           cost[899]/cost[0], cost[999]/cost[0])
    TT_bce.append(cost_bce)
    
###################### Ours(F1) ############################## 
    start_f1 = time.time()
    theta = np.zeros(X_train_a.shape[1])
    num_iter = 1000
    lr = 0.001
    beta = 1            
    L = 73               
    cost_f1 = []
    bs = beta**2
    sy = np.sum(y_train)
    bs_sy = bs*sy
    for i in range(num_iter):
        p = sigmoid(X_train_a,theta)
        yh = sigmoid_2(p)                                          
        syh = np.sum(yh)
        syhy = np.dot(yh,y_train)
        cost_f1.append(cost_function_new(bs,syh,syhy,bs_sy))                
        gradient = gradient_descent_new(p,X_train_a,yh,y_train,bs,syh,syhy,bs_sy)      
        theta = update_loss(theta,lr,gradient)
#     plot_cost_function(cost)
#     print(cost[0]/cost[0], cost[99]/cost[0], cost[199]/cost[0], cost[299]/cost[0], cost[399]/cost[0]
#          , cost[499]/cost[0], cost[599]/cost[0], cost[699]/cost[0], cost[799]/cost[0], 
#           cost[899]/cost[0], cost[999]/cost[0])
    TT_f1.append(cost_f1)
    
###################### Ours(Gmean) ##############################
    start_gmean = time.time()
    theta = np.zeros(X_train_a.shape[1])
    num_iter = 1000
    lr = 0.003         
    L = 73               
    cost_gmean = []
    sy = np.sum(y_train)
    yl = y_train.shape[0]
    for i in range(num_iter):
        p = sigmoid(X_train_a,theta)
        yh = sigmoid_2(p)                                          
        syh = np.sum(yh)
        syhy = np.dot(yh,y_train)
        cost_gmean.append(cost_function_gmean(sy,syh,syhy,yl))                
        gradient = gradient_descent_gmean(p,X_train_a,yh,y_train,sy,syh,syhy,yl)      
        theta = update_loss(theta,lr,gradient)
#     plot_cost_function(cost)
#     print(cost[0]/cost[0], cost[99]/cost[0], cost[199]/cost[0], cost[299]/cost[0], cost[399]/cost[0]
#          , cost[499]/cost[0], cost[599]/cost[0], cost[699]/cost[0], cost[799]/cost[0], 
#           cost[899]/cost[0], cost[999]/cost[0])
    TT_gmean.append(cost_gmean)
    
###################### Ours(Balanced Accuracy) ##############################
    start_bacc = time.time()
    theta = np.zeros(X_train_a.shape[1])
    num_iter = 1000
    lr = 0.005
    L = 73               
    cost_bacc = []
    sy = np.sum(y_train)
    yl = y_train.shape[0]
    for i in range(num_iter):
        p = sigmoid(X_train_a,theta)
        yh = sigmoid_2(p)                                          
        syh = np.sum(yh)
        syhy = np.dot(yh,y_train)
        cost_bacc.append(cost_function_balacc(sy,syh,syhy,yl))                
        gradient = gradient_descent_balacc(p,X_train_a,yh,y_train,sy,yl)     
        theta = update_loss(theta,lr,gradient)
#     plot_cost_function(cost)
#     print(cost[0]/cost[0], cost[99]/cost[0], cost[199]/cost[0], cost[299]/cost[0], cost[399]/cost[0]
#          , cost[499]/cost[0], cost[599]/cost[0], cost[699]/cost[0], cost[799]/cost[0], 
#           cost[899]/cost[0], cost[999]/cost[0])
    TT_bacc.append(cost_bacc)

In [ ]:
tts = [TT_bce, TT_f1, TT_gmean, TT_bacc]
cost_bce=[]
cost_f1=[]
cost_gmean=[]
cost_bacc=[]
ars = [cost_bce, cost_f1, cost_gmean, cost_bacc]
for k in range(4):
    for i in range(1000):
        suum = 0
        for j in range(10):
            suum += tts[k][j][i]
        ars[k].append(suum/10)

In [ ]:
# plt.rcParams["figure.figsize"] = [5, 3]
plt.rcParams["figure.dpi"] = 200
plt.rcParams['axes.titlesize'] = 20  
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['axes.labelsize'] = 20  
plt.rcParams['font.size'] = 15

# plt.plot(cost_mse,label="MSE", color='green')
plt.plot(cost_bce,label="BCE", color='red', linewidth = 3)
plt.plot(cost_f1,label="$L_{F1}$", color='deepskyblue', ls='-', linewidth = 3)
plt.plot(cost_gmean,label="$L_{G}$", color='deepskyblue', ls=':', linewidth = 3)
plt.plot(cost_bacc,label="$L_{B}$", color='deepskyblue', ls='--', linewidth = 3)

plt.ylim(-0.1,1.1)
plt.xlabel('Iteration',fontsize = 20)
plt.ylabel('Loss',fontsize = 20)
plt.title("Loss Curve (Data#3)",fontsize = 20)
plt.legend()
plt.show()  

# 4. Diabetes Prediction Data (8d / 100000)

In [ ]:
# class 'B' = Benign, class 'M' = Malignant
diab_df = pd.read_csv('diabetes_prediction_dataset.csv')
diab_df.shape

In [ ]:
diab_df.isnull().sum()

In [ ]:
diab_df

In [ ]:
# Female = 0, Male = 1, other = 2
gen_encoded, gen_class = pd.factorize(diab_df['gender'])
print(gen_class)
gen_encoded

In [ ]:
# Female = 0, Male = 1, other = 2
pd.Series(gen_encoded).value_counts()

In [ ]:
diab_df['gender'] = gen_encoded
diab_df

In [ ]:
# never = 0, Info = 1, current = 2, former=3, ever=4, not current=5
smo_encoded, smo_class = pd.factorize(diab_df['smoking_history'])
print(smo_class)
smo_encoded

In [ ]:
# never = 0, Info = 1, current = 2, former=3, ever=4, not current=5
pd.Series(smo_encoded).value_counts()

In [ ]:
diab_df['smoking_history'] = smo_encoded
diab_df

In [ ]:
diab_df.describe()

In [ ]:
# Amount values largely varies.

# # Normalization
# card_df.iloc[:,:-1] = (card_df.iloc[:,:-1] - card_df.iloc[:,:-1].min())/(card_df.iloc[:,:-1].max() - card_df.iloc[:,:-1].min())

# Standardization
diab_df.iloc[:,:-1] = (diab_df.iloc[:,:-1] - diab_df.iloc[:,:-1].mean())/diab_df.iloc[:,:-1].std()

diab_df

In [ ]:
# imbalanced Data, very good
# None = 0, Diabetes = 1
diab_df['diabetes'].value_counts()

In [ ]:
TT_mse= []
TT_bce= []
TT_f1= []
TT_gmean= []
TT_bacc= []

In [ ]:
# train = train data // test = validation data // 10 cross-validation(train 0.9, validate 0.1, 10 times)
# The average of test results are the validation scores.

from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state = 2)
n_iter=0

X = diab_df.iloc[:, :-1]
y = diab_df.iloc[:, -1]


for train_index, test_index in skf.split(diab_df, diab_df['diabetes']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
#     print('Labels for train:\n', label_train.value_counts())
#     print('Labels for test:\n', label_test.value_counts())
#     print(len(X_train), len(y_train), len(X_test), len(y_test))
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    X_test = np.array(X_test)
    y_test = np.array(y_test)    
    intercept_r = np.ones((X_train.shape[0],1))
    intercept_t = np.ones((X_test.shape[0],1))
    X_train_a = np.concatenate((intercept_r,X_train),axis= 1)
    X_test_a = np.concatenate((intercept_t,X_test),axis= 1)
        
###################### Label change (-1 -> 0) ############################## 
    y_train = [0 if x==-1 else x for x in y_train]
    y_train = np.array(y_train)
    y_test = [0 if x==-1 else x for x in y_test]
    y_test = np.array(y_test)
    
###################### BCE (with Sigmoid) ############################## 
    start_bce = time.time()
    theta = np.zeros(X_train_a.shape[1])
    threshold = 0.5
    num_iter = 1000
    cost_bce = []
    lr = 0.1
    yl = y_train.shape[0]
    for i in range(num_iter):
        h = sigmoid(X_train_a,theta)
        cost_bce.append(cost_function(h,y_train))
        gradient = gradient_descent(X_train_a,h,y_train,yl)
        theta = update_loss(theta,lr,gradient)
#     plot_cost_function(cost)
#     print(cost[0]/cost[0], cost[99]/cost[0], cost[199]/cost[0], cost[299]/cost[0], cost[399]/cost[0]
#          , cost[499]/cost[0], cost[599]/cost[0], cost[699]/cost[0], cost[799]/cost[0], 
#           cost[899]/cost[0], cost[999]/cost[0])
    TT_bce.append(cost_bce)
    
###################### Ours(F1) ############################## 
    start_f1 = time.time()
    theta = np.zeros(X_train_a.shape[1])
    num_iter = 1000
    lr = 0.01
    beta = 1            
    L = 73               
    cost_f1 = []
    bs = beta**2
    sy = np.sum(y_train)
    bs_sy = bs*sy
    for i in range(num_iter):
        p = sigmoid(X_train_a,theta)
        yh = sigmoid_2(p)                                          
        syh = np.sum(yh)
        syhy = np.dot(yh,y_train)
        cost_f1.append(cost_function_new(bs,syh,syhy,bs_sy))                
        gradient = gradient_descent_new(p,X_train_a,yh,y_train,bs,syh,syhy,bs_sy)      
        theta = update_loss(theta,lr,gradient)
#     plot_cost_function(cost)
#     print(cost[0]/cost[0], cost[99]/cost[0], cost[199]/cost[0], cost[299]/cost[0], cost[399]/cost[0]
#          , cost[499]/cost[0], cost[599]/cost[0], cost[699]/cost[0], cost[799]/cost[0], 
#           cost[899]/cost[0], cost[999]/cost[0])
    TT_f1.append(cost_f1)
    
###################### Ours(Gmean) ##############################
    start_gmean = time.time()
    theta = np.zeros(X_train_a.shape[1])
    num_iter = 1000
    lr = 0.01          
    L = 73               
    cost_gmean = []
    sy = np.sum(y_train)
    yl = y_train.shape[0]
    for i in range(num_iter):
        p = sigmoid(X_train_a,theta)
        yh = sigmoid_2(p)                                          
        syh = np.sum(yh)
        syhy = np.dot(yh,y_train)
        cost_gmean.append(cost_function_gmean(sy,syh,syhy,yl))                
        gradient = gradient_descent_gmean(p,X_train_a,yh,y_train,sy,syh,syhy,yl)      
        theta = update_loss(theta,lr,gradient)
#     plot_cost_function(cost)
#     print(cost[0]/cost[0], cost[99]/cost[0], cost[199]/cost[0], cost[299]/cost[0], cost[399]/cost[0]
#          , cost[499]/cost[0], cost[599]/cost[0], cost[699]/cost[0], cost[799]/cost[0], 
#           cost[899]/cost[0], cost[999]/cost[0])
    TT_gmean.append(cost_gmean)
    
###################### Ours(Balanced Accuracy) ##############################
    start_bacc = time.time()
    theta = np.zeros(X_train_a.shape[1])
    num_iter = 1000
    lr = 0.01
    L = 73               
    cost_bacc = []
    sy = np.sum(y_train)
    yl = y_train.shape[0]
    for i in range(num_iter):
        p = sigmoid(X_train_a,theta)
        yh = sigmoid_2(p)                                          
        syh = np.sum(yh)
        syhy = np.dot(yh,y_train)
        cost_bacc.append(cost_function_balacc(sy,syh,syhy,yl))                
        gradient = gradient_descent_balacc(p,X_train_a,yh,y_train,sy,yl)     
        theta = update_loss(theta,lr,gradient)
#     plot_cost_function(cost)
#     print(cost[0]/cost[0], cost[99]/cost[0], cost[199]/cost[0], cost[299]/cost[0], cost[399]/cost[0]
#          , cost[499]/cost[0], cost[599]/cost[0], cost[699]/cost[0], cost[799]/cost[0], 
#           cost[899]/cost[0], cost[999]/cost[0])
    TT_bacc.append(cost_bacc)

In [ ]:
tts = [TT_bce, TT_f1, TT_gmean, TT_bacc]
cost_bce=[]
cost_f1=[]
cost_gmean=[]
cost_bacc=[]
ars = [cost_bce, cost_f1, cost_gmean, cost_bacc]
for k in range(4):
    for i in range(1000):
        suum = 0
        for j in range(10):
            suum += tts[k][j][i]
        ars[k].append(suum/10)

In [ ]:
# plt.rcParams["figure.figsize"] = [5, 3]
plt.rcParams["figure.dpi"] = 200
plt.rcParams['axes.titlesize'] = 20  
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['axes.labelsize'] = 20  
plt.rcParams['font.size'] = 15

# plt.plot(cost_mse,label="MSE", color='green')
plt.plot(cost_bce,label="BCE", color='red', linewidth = 3)
plt.plot(cost_f1,label="$L_{F1}$", color='deepskyblue', ls='-', linewidth = 3)
plt.plot(cost_gmean,label="$L_{G}$", color='deepskyblue', ls=':', linewidth = 3)
plt.plot(cost_bacc,label="$L_{B}$", color='deepskyblue', ls='--', linewidth = 3)

plt.ylim(-0.1,1.1)
plt.xlabel('Iteration',fontsize = 20)
plt.ylabel('Loss',fontsize = 20)
plt.title("Loss Curve (Data#4)",fontsize = 20)
plt.legend()
plt.show()  